# Task 4



In [1]:
!pip install transformer-lens transformers accelerate einops pandas matplotlib plotly circuitsvis

  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch
import transformer_lens
import transformer_lens.utils as utils
from transformer_lens import HookedTransformer

import plotly.express as px
import plotly.io as pio
import circuitsvis as cv

torch.set_grad_enabled(False) # Turn off gradients for interpretability
print("Libraries imported successfully.")

/tmp/ipykernel_506/3895170662.py:3: DeprecationWarning: The 'utils' module has been deprecated. Please use 'transformer_lens.utilities' instead. Importing from utils.py will be removed in TransformerLens 4.0.
  import transformer_lens.utils as utils


Libraries imported successfully.


## 1. Load the Model
We load `succinctly/text2image-prompt-generator` into `HookedTransformer`. This model uses GPT-2 as its base architecture.

In [3]:
MODEL_NAME = "succinctly/text2image-prompt-generator"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [4]:
from transformer_lens import HookedTransformer

try:
    model = HookedTransformer.from_pretrained(MODEL_NAME, device=device)
except Exception:
    from transformer_lens.model_bridge import TransformerBridge
    model = TransformerBridge.boot_transformers(MODEL_NAME, device=device)
    model.enable_compatibility_mode(disable_warnings=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/907 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/665M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/665M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

In [5]:
print(type(model))
print("Layers:", model.cfg.n_layers)
print("Heads:", model.cfg.n_heads)
print("d_model:", model.cfg.d_model)
print("d_vocab:", model.cfg.d_vocab)

<class 'transformer_lens.model_bridge.bridge.TransformerBridge'>
Layers: 12
Heads: 12
d_model: 768
d_vocab: 50257


In [6]:
logits = model("the black man looks like a ")
next_token_logits = logits[0, -1]
next_token_prediction = next_token_logits.argmax()
next_word_prediction = model.tokenizer.decode(next_token_prediction)
print(next_word_prediction)

 cat


In [7]:
logits, cache = model.run_with_cache("a blue sports car looks like a")
for key, value in cache.items():
    print(key, value.shape)

embed.hook_in torch.Size([1, 8])
hook_embed torch.Size([1, 8, 768])
embed.hook_out torch.Size([1, 8, 768])
pos_embed.hook_in torch.Size([1, 8])
hook_pos_embed torch.Size([1, 8, 768])
pos_embed.hook_out torch.Size([1, 8, 768])
blocks.0.hook_in torch.Size([1, 8, 768])
blocks.0.hook_resid_pre torch.Size([1, 8, 768])
blocks.0.ln1.hook_in torch.Size([1, 8, 768])
blocks.0.ln1.hook_scale torch.Size([1, 8, 1])
blocks.0.ln1.hook_normalized torch.Size([1, 8, 768])
blocks.0.ln1.hook_out torch.Size([1, 8, 768])
blocks.0.attn.hook_in torch.Size([1, 8, 768])
blocks.0.attn.q.hook_in torch.Size([1, 8, 768])
blocks.0.attn.hook_q torch.Size([1, 8, 12, 64])
blocks.0.attn.q.hook_out torch.Size([1, 8, 12, 64])
blocks.0.attn.k.hook_in torch.Size([1, 8, 768])
blocks.0.attn.hook_k torch.Size([1, 8, 12, 64])
blocks.0.attn.k.hook_out torch.Size([1, 8, 12, 64])
blocks.0.attn.v.hook_in torch.Size([1, 8, 768])
blocks.0.attn.hook_v torch.Size([1, 8, 12, 64])
blocks.0.attn.v.hook_out torch.Size([1, 8, 12, 64])
block

In [8]:
for key, value in cache.items():
    if "pattern" in key:
        print(key, value.shape)

blocks.0.attn.hook_pattern torch.Size([1, 12, 8, 8])
blocks.1.attn.hook_pattern torch.Size([1, 12, 8, 8])
blocks.2.attn.hook_pattern torch.Size([1, 12, 8, 8])
blocks.3.attn.hook_pattern torch.Size([1, 12, 8, 8])
blocks.4.attn.hook_pattern torch.Size([1, 12, 8, 8])
blocks.5.attn.hook_pattern torch.Size([1, 12, 8, 8])
blocks.6.attn.hook_pattern torch.Size([1, 12, 8, 8])
blocks.7.attn.hook_pattern torch.Size([1, 12, 8, 8])
blocks.8.attn.hook_pattern torch.Size([1, 12, 8, 8])
blocks.9.attn.hook_pattern torch.Size([1, 12, 8, 8])
blocks.10.attn.hook_pattern torch.Size([1, 12, 8, 8])
blocks.11.attn.hook_pattern torch.Size([1, 12, 8, 8])


In [46]:
utils.test_prompt("A cinematic shot of a red car", " red", model)

Tokenized prompt: ['<|endoftext|>', 'A', ' cinematic', ' shot', ' of', ' a', ' red', ' car']
Tokenized answer: [' red']


Performance on answer token:
Rank: 343      Logit: 10.92 Prob:  0.00% Token: | red|

Top 0th token. Logit: 20.16 Prob: 39.91% Token: | in|
Top 1th token. Logit: 19.31 Prob: 17.11% Token: | with|
Top 2th token. Logit: 18.71 Prob:  9.38% Token: |,|
Top 3th token. Logit: 18.13 Prob:  5.27% Token: | on|
Top 4th token. Logit: 17.95 Prob:  4.37% Token: | covered|
Top 5th token. Logit: 17.42 Prob:  2.59% Token: | surrounded|
Top 6th token. Logit: 17.36 Prob:  2.42% Token: | made|
Top 7th token. Logit: 16.94 Prob:  1.59% Token: | at|
Top 8th token. Logit: 16.86 Prob:  1.48% Token: | driving|
Top 9th token. Logit: 16.63 Prob:  1.18% Token: | as|


Ranks of the answer tokens: [(' red', 343)]

In [9]:
utils.test_prompt("A cinematic shot of a red car",
                   " red", model)

utils.test_prompt("A cinematic shot of a blue car",
                   " blue", model)

Tokenized prompt: ['<|endoftext|>', 'A', ' cinematic', ' shot', ' of', ' a', ' red', ' car']
Tokenized answer: [' red']


Performance on answer token:
Rank: 343      Logit: 10.92 Prob:  0.00% Token: | red|

Top 0th token. Logit: 20.16 Prob: 39.91% Token: | in|
Top 1th token. Logit: 19.31 Prob: 17.11% Token: | with|
Top 2th token. Logit: 18.71 Prob:  9.38% Token: |,|
Top 3th token. Logit: 18.13 Prob:  5.27% Token: | on|
Top 4th token. Logit: 17.95 Prob:  4.37% Token: | covered|
Top 5th token. Logit: 17.42 Prob:  2.59% Token: | surrounded|
Top 6th token. Logit: 17.36 Prob:  2.42% Token: | made|
Top 7th token. Logit: 16.94 Prob:  1.59% Token: | at|
Top 8th token. Logit: 16.86 Prob:  1.48% Token: | driving|
Top 9th token. Logit: 16.63 Prob:  1.18% Token: | as|


Ranks of the answer tokens: [(' red', 343)]

Tokenized prompt: ['<|endoftext|>', 'A', ' cinematic', ' shot', ' of', ' a', ' blue', ' car']
Tokenized answer: [' blue']


Performance on answer token:
Rank: 1029     Logit:  9.36 Prob:  0.00% Token: | blue|

Top 0th token. Logit: 20.20 Prob: 31.77% Token: | in|
Top 1th token. Logit: 19.32 Prob: 13.18% Token: |,|
Top 2th token. Logit: 19.30 Prob: 12.99% Token: | with|
Top 3th token. Logit: 18.54 Prob:  6.08% Token: | made|
Top 4th token. Logit: 18.41 Prob:  5.31% Token: | covered|
Top 5th token. Logit: 18.34 Prob:  4.97% Token: | on|
Top 6th token. Logit: 17.83 Prob:  2.98% Token: | surrounded|
Top 7th token. Logit: 17.29 Prob:  1.74% Token: | as|
Top 8th token. Logit: 17.12 Prob:  1.47% Token: | at|
Top 9th token. Logit: 17.12 Prob:  1.47% Token: | driving|


Ranks of the answer tokens: [(' blue', 1029)]


corrupt prompt

In [24]:
prompt_red = "A photo of a red car. The color of the car is"
prompt_blue = "A photo of a blue car. The color of the car is"

tokens_red = model.to_str_tokens(prompt_red)
tokens_blue = model.to_str_tokens(prompt_blue)

print("Red Prompt Tokens:", tokens_red)
print("Blue Prompt Tokens:", tokens_blue)

assert len(tokens_red) == len(tokens_blue), (
    f"Token count mismatch ({len(tokens_red)} vs {len(tokens_blue)}) - "
    "position-aligned patching requires equal-length sequences."
)

color_idx = tokens_red.index(" red")
color_idx_blue = tokens_blue.index(" blue")
print(f"\nThe color token is at index: {color_idx} (red) / {color_idx_blue} (blue)")

Red Prompt Tokens: ['<|endoftext|>', 'A', ' photo', ' of', ' a', ' red', ' car', '.', ' The', ' color', ' of', ' the', ' car', ' is']
Blue Prompt Tokens: ['<|endoftext|>', 'A', ' photo', ' of', ' a', ' blue', ' car', '.', ' The', ' color', ' of', ' the', ' car', ' is']

The color token is at index: 5 (red) / 5 (blue)


In [25]:
utils.test_prompt(prompt_red, " red", model)
utils.test_prompt(prompt_blue, " blue", model)

Tokenized prompt: ['<|endoftext|>', 'A', ' photo', ' of', ' a', ' red', ' car', '.', ' The', ' color', ' of', ' the', ' car', ' is']
Tokenized answer: [' red']


Performance on answer token:
Rank: 0        Logit: 16.17 Prob: 14.34% Token: | red|

Top 0th token. Logit: 16.17 Prob: 14.34% Token: | red|
Top 1th token. Logit: 16.13 Prob: 13.68% Token: | orange|
Top 2th token. Logit: 15.94 Prob: 11.32% Token: | blue|
Top 3th token. Logit: 15.54 Prob:  7.59% Token: | green|
Top 4th token. Logit: 15.21 Prob:  5.50% Token: | white|
Top 5th token. Logit: 15.14 Prob:  5.10% Token: | black|
Top 6th token. Logit: 14.86 Prob:  3.87% Token: | yellow|
Top 7th token. Logit: 14.69 Prob:  3.26% Token: | a|
Top 8th token. Logit: 14.41 Prob:  2.46% Token: | pink|
Top 9th token. Logit: 14.23 Prob:  2.06% Token: | purple|


Ranks of the answer tokens: [(' red', 0)]

Tokenized prompt: ['<|endoftext|>', 'A', ' photo', ' of', ' a', ' blue', ' car', '.', ' The', ' color', ' of', ' the', ' car', ' is']
Tokenized answer: [' blue']


Performance on answer token:
Rank: 0        Logit: 17.08 Prob: 25.81% Token: | blue|

Top 0th token. Logit: 17.08 Prob: 25.81% Token: | blue|
Top 1th token. Logit: 16.52 Prob: 14.65% Token: | orange|
Top 2th token. Logit: 15.90 Prob:  7.91% Token: | green|
Top 3th token. Logit: 15.59 Prob:  5.81% Token: | white|
Top 4th token. Logit: 15.29 Prob:  4.29% Token: | red|
Top 5th token. Logit: 15.14 Prob:  3.71% Token: | black|
Top 6th token. Logit: 14.98 Prob:  3.17% Token: | yellow|
Top 7th token. Logit: 14.94 Prob:  3.02% Token: | a|
Top 8th token. Logit: 14.67 Prob:  2.31% Token: | pink|
Top 9th token. Logit: 14.65 Prob:  2.27% Token: | purple|


Ranks of the answer tokens: [(' blue', 0)]

##  Activation Caching


In [26]:
logits_red, cache_red = model.run_with_cache(prompt_red)
logits_blue, cache_blue = model.run_with_cache(prompt_blue)

print(f"Cached {len(cache_red)} internal tensors!")

Cached 493 internal tensors!


## attention analysis

In [49]:
# Pick a layer to inspect (usually middle to late layers do the heavy conceptual lifting)
layer_to_inspect = 3

attention_pattern = cache_red["pattern", layer_to_inspect]

# Visualize using CircuitsVis
print(f"Attention patterns for Layer {layer_to_inspect}:")
cv.attention.attention_patterns(
    tokens=tokens_red,
    attention=attention_pattern[0]
)

Attention patterns for Layer 3:


In [30]:
x3final_token_idx = -1
head_scores = []

for layer in range(model.cfg.n_layers):
    attention_pattern = cache_red["pattern", layer][0]
    for head in range(model.cfg.n_heads):
        score = attention_pattern[head, final_token_idx, color_idx].item()
        head_scores.append((layer, head, score))

head_scores.sort(key=lambda x: x[2], reverse=True)

print("Top 5 'Visual Attribute Fetcher' Heads:")
print("-" * 40)
for i in range(5):
    layer, head, score = head_scores[i]
    print(f"Layer {layer}, Head {head:2d}  |  Attention Weight: {score:.4f}")

Top 5 'Visual Attribute Fetcher' Heads:
----------------------------------------
Layer 8, Head 11  |  Attention Weight: 0.1998
Layer 9, Head  9  |  Attention Weight: 0.1940
Layer 5, Head 10  |  Attention Weight: 0.1571
Layer 8, Head  3  |  Attention Weight: 0.1484
Layer 11, Head 10  |  Attention Weight: 0.1306


In [50]:
import torch
from functools import partial



# Clean and corrupt prompts in variables
prompt_red = "A photo of a red car. The color of the car is"
prompt_blue = "A photo of a blue car. The color of the car is"

# Get the list of tokens the model will deal with
tokens_red = model.to_str_tokens(prompt_red)
# Indices of the right and wrong answers (colors) to judge what the model predicts
_, cache_blue = model.run_with_cache(prompt_blue)

def patch_residual_stream(activations, hook, layer="blocks.6.hook_resid_post", pos=5):
   # The residual stream dimensions are [batch, position, d_embed]
   activations[:, pos, :] = cache_blue[layer][:, pos, :]
   return activations

# List of layers and positions to iterate over. We want to patch before the
# first layer, and after every layer.
layers = ["blocks.0.hook_resid_pre", *[f"blocks.{i}.hook_resid_post" for i in range(model.cfg.n_layers)]]
n_layers = len(layers)
n_pos = len(tokens_red)

# Indices of the right and wrong answers (colors) to judge what the model predicts
clean_answer_index = model.to_tokens(prompt_red)[0, tokens_red.index(" red")].item()
corrupt_answer_index = model.to_tokens(prompt_blue)[0, model.to_str_tokens(prompt_blue).index(" blue")].item()

# Test the effect of this patch at any layer and any position
patching_effect = torch.zeros(n_layers, n_pos)
for l, layer in enumerate(layers):
    print("Patching layer", l)
    for pos in range(n_pos):
        fwd_hooks = [(layer, partial(patch_residual_stream, layer=layer, pos=pos))]
        prediction_logits = model.run_with_hooks(prompt_red,
                                                    fwd_hooks=fwd_hooks)[0, -1]
        patching_effect[l, pos] = prediction_logits[clean_answer_index] \
                                  - prediction_logits[corrupt_answer_index]

torch.cuda.empty_cache()

Patching layer 0
Patching layer 1
Patching layer 2
Patching layer 3
Patching layer 4
Patching layer 5
Patching layer 6
Patching layer 7
Patching layer 8
Patching layer 9
Patching layer 10
Patching layer 11
Patching layer 12


In [33]:
# Plot
token_labels = [f"(pos {i:2}) {t}" for i, t in enumerate(tokens_red)]

fig = px.imshow(
    patching_effect.numpy(),
    labels=dict(x="pos", y="layer", color="Logit difference"),
    x=token_labels,
    y=layers,
    title="Patching with color word",
    width=600,
    height=380,
)
fig.show()

In [43]:
from functools import partial

# hook_result is only populated if this flag is set — must be set BEFORE run_with_cache
model.set_use_attn_result(True)

# Re-cache the corrupt prompt now that hook_result will actually be recorded
_, cache_blue = model.run_with_cache(prompt_blue)

n_layers = model.cfg.n_layers
n_heads = model.cfg.n_heads

def patch_head_result(activations, hook, head=None):
    # activations: [batch, position, head_index, d_model]
    activations[:, :, head, :] = cache_blue[hook.name][:, :, head, :]
    return activations

patching_effect = torch.zeros(n_layers, n_heads)
for layer in range(n_layers):
    print("Patching layer", layer, "/", n_layers - 1)
    for head in range(n_heads):
        fwd_hooks = [(f"blocks.{layer}.attn.hook_result", partial(patch_head_result, head=head))]
        prediction_logits = model.run_with_hooks(prompt_red, fwd_hooks=fwd_hooks)[0, -1]
        patching_effect[layer, head] = (
            prediction_logits[clean_answer_index] - prediction_logits[corrupt_answer_index]
        ).item()

torch.cuda.empty_cache()

Patching layer 0 / 11
Patching layer 1 / 11
Patching layer 2 / 11
Patching layer 3 / 11
Patching layer 4 / 11
Patching layer 5 / 11
Patching layer 6 / 11
Patching layer 7 / 11
Patching layer 8 / 11
Patching layer 9 / 11
Patching layer 10 / 11
Patching layer 11 / 11


In [44]:
fig = px.imshow(
    patching_effect.numpy(),
    labels=dict(x="Head", y="Layer", color="Logit difference"),
    x=[str(h) for h in range(n_heads)],
    y=[str(l) for l in range(n_layers)],
    title="Per-head patching: which head causally carries 'red' vs 'blue'?",
    color_continuous_scale="RdBu",
    aspect="auto",
)
fig.show()